# Sprint 4 — Webinar 13 (Teórico)
## Data Journey → Funnels → Cohorts con SQL

**Programa:** Data Analytics
**Sprint:** 4
**Duración:** 100 minutos
**Modalidad:** Teórica con ejercicios prácticos en SQL
**Motor SQL:** [SQL Workbench](https://sql-workbench.com/) (DuckDB en el navegador)

> Esta clase cubre el viaje del usuario (Data Journey), el uso de CTEs para estructurar consultas, y la construcción de funnels de conversión y análisis de cohorts — todo desde SQL puro.

<div style="text-align: center">
    <img src="https://raw.githubusercontent.com/ljpiere/tpdata_python/main/images/w1s1_2.png" width="400">
</div>

## Agenda sugerida (100 minutos)

| Tiempo | Bloque | Contenido |
|-------:|--------|-----------|
| 0–5 | Apertura | Bienvenida, objetivos y setup de SQL Workbench |
| 5–20 | Bloque 1 | ¿Qué es el Data Journey? Concepto, etapas, métricas y campos clave |
| 20–40 | Bloque 2 | ¿Qué son las CTEs? Sintaxis, ventajas y ejercicios progresivos |
| 40–60 | Bloque 3 | Construcción de Funnels de conversión con CTEs |
| 60–80 | Bloque 4 | Análisis de retención con Cohorts |
| 80–95 | Bloque 5 | Ejercicios integradores (dificultad creciente) |
| 95–100 | Cierre | Takeaways, Q&A y próximos pasos |

## Objetivos de aprendizaje

Al finalizar la sesión, el/la estudiante podrá:

1. **Definir** qué es un *Data Journey* (User Journey) y por qué es fundamental para productos digitales.
2. **Identificar** las etapas típicas de un journey y las métricas asociadas (conversión, drop-off, retención).
3. **Reconocer** los campos clave en una base de datos de eventos (`user_id`, `event_ts`, `event_name`, `props`).
4. **Explicar** qué son las CTEs (Common Table Expressions), su sintaxis y sus ventajas frente a subconsultas.
5. **Usar** CTEs para organizar consultas complejas paso a paso.
6. **Construir** un funnel de conversión completo con CTEs, calculando drop-off y tasas de conversión.
7. **Agrupar** usuarios por cohort (mensual) y construir tablas de retención.
8. **Segmentar** la retención por atributos de negocio (plan, canal, dispositivo).

## Setup: Motor SQL

Usaremos **[SQL Workbench](https://sql-workbench.com/)** directamente en el navegador. Este motor utiliza **DuckDB**, que es compatible con la mayoría de la sintaxis SQL estándar.

**Instrucciones:**
1. Abre [https://sql-workbench.com/](https://sql-workbench.com/) en tu navegador.
2. Copia y pega los bloques SQL de este notebook en el editor.
3. Ejecuta con el botón ▶ o `Ctrl+Enter`.

> **Nota:** Cada bloque SQL de esta clase está listo para ejecutarse directamente en SQL Workbench. Empieza siempre por el bloque de creación de tablas (Bloque 0).

---
# Bloque 0 · Creación de tablas y datos de práctica

> **Ejecuta este bloque completo primero** en SQL Workbench. Crea las tablas `users` y `events` con datos de un e-commerce ficticio (enero–febrero 2021).

### 0.1 Creación de tablas (DDL)

```sql
-- =============================================
-- DDL: Tablas para Journey / Funnel / Cohorts
-- Compatible con DuckDB (SQL Workbench)
-- =============================================

-- Tabla de usuarios con atributos de segmentación
CREATE OR REPLACE TABLE users (
  user_id     INTEGER PRIMARY KEY,
  signup_ts   TIMESTAMP NOT NULL,
  plan        VARCHAR NOT NULL,     -- 'free' o 'paid'
  channel     VARCHAR NOT NULL,     -- 'organic', 'ads', 'referral', 'email'
  device      VARCHAR NOT NULL      -- 'web', 'android', 'ios'
);

-- Tabla de eventos tipo GA4 minimalista
CREATE OR REPLACE TABLE events (
  event_id    INTEGER PRIMARY KEY,
  user_id     INTEGER NOT NULL,
  event_name  VARCHAR NOT NULL,
  event_ts    TIMESTAMP NOT NULL,
  props       VARCHAR NOT NULL DEFAULT '{}'
);
```

### 0.2 Carga de datos (Seed)

```sql
-- =============================================
-- SEED: Usuarios (cohortes enero y febrero 2021)
-- =============================================
INSERT INTO users (user_id, signup_ts, plan, channel, device) VALUES
  (1,  '2021-01-02 09:10:00', 'free',  'organic',  'web'),
  (2,  '2021-01-03 14:20:00', 'paid',  'ads',      'ios'),
  (3,  '2021-01-05 08:05:00', 'free',  'referral', 'android'),
  (4,  '2021-01-10 18:42:00', 'free',  'organic',  'web'),
  (5,  '2021-01-12 11:11:00', 'paid',  'email',    'web'),
  (6,  '2021-02-01 09:15:00', 'free',  'ads',      'android'),
  (7,  '2021-02-05 16:00:00', 'paid',  'referral', 'ios'),
  (8,  '2021-02-09 10:30:00', 'free',  'organic',  'web'),
  (9,  '2021-02-12 21:45:00', 'free',  'email',    'android'),
  (10, '2021-02-15 07:50:00', 'paid',  'ads',      'web'),
  -- Usuarios extra para cohort de marzo
  (11, '2021-03-01 10:00:00', 'free',  'organic',  'web'),
  (12, '2021-03-03 12:30:00', 'paid',  'ads',      'ios'),
  (13, '2021-03-08 09:15:00', 'free',  'referral', 'android'),
  (14, '2021-03-12 14:00:00', 'paid',  'email',    'web'),
  (15, '2021-03-18 17:45:00', 'free',  'organic',  'android');

-- =============================================
-- SEED: Eventos
-- Flujo: session_start → view_item → add_to_cart → begin_checkout → purchase
-- =============================================

-- Enero (usuarios 1..5)
INSERT INTO events (event_id, user_id, event_name, event_ts, props) VALUES
  (1,  1, 'session_start',  '2021-01-02 09:12:00', '{"utm":"organic"}'),
  (2,  1, 'view_item',      '2021-01-02 09:14:00', '{"sku":"A101"}'),
  (3,  1, 'add_to_cart',    '2021-01-02 09:15:00', '{"sku":"A101"}'),
  (4,  1, 'begin_checkout', '2021-01-02 09:16:00', '{"cart_value":29.9}'),
  (5,  1, 'purchase',       '2021-01-02 09:18:00', '{"order_id":"O-0001","value":29.9}'),

  (6,  2, 'session_start',  '2021-01-03 14:21:00', '{"utm":"ads"}'),
  (7,  2, 'view_item',      '2021-01-03 14:22:00', '{"sku":"B222"}'),
  (8,  2, 'add_to_cart',    '2021-01-03 14:23:00', '{"sku":"B222"}'),
  (9,  2, 'begin_checkout', '2021-01-03 14:24:00', '{"cart_value":58.0}'),
  (10, 2, 'purchase',       '2021-01-03 14:27:00', '{"order_id":"O-0002","value":58.0}'),

  (11, 3, 'session_start',  '2021-01-05 08:06:00', '{"utm":"referral"}'),
  (12, 3, 'view_item',      '2021-01-05 08:07:00', '{"sku":"C333"}'),
  (13, 3, 'add_to_cart',    '2021-01-05 08:08:00', '{"sku":"C333"}'),
  (14, 3, 'begin_checkout', '2021-01-05 08:09:00', '{"cart_value":105.0}'),
  -- Usuario 3 NO compra (abandona en checkout)

  (15, 4, 'session_start',  '2021-01-10 18:43:00', '{"utm":"organic"}'),
  (16, 4, 'view_item',      '2021-01-10 18:46:00', '{"sku":"D444"}'),
  -- Usuario 4 solo mira un producto

  (17, 5, 'session_start',  '2021-01-12 11:12:00', '{"utm":"email"}'),
  (18, 5, 'view_item',      '2021-01-12 11:13:00', '{"sku":"A101"}'),
  (19, 5, 'add_to_cart',    '2021-01-12 11:14:00', '{"sku":"A101"}');
  -- Usuario 5 abandona el carrito

-- Febrero (usuarios 6..10)
INSERT INTO events (event_id, user_id, event_name, event_ts, props) VALUES
  (20, 6, 'session_start',  '2021-02-01 09:17:00', '{"utm":"ads"}'),
  (21, 6, 'view_item',      '2021-02-01 09:18:00', '{"sku":"E555"}'),
  -- Usuario 6 solo mira

  (22, 7, 'session_start',  '2021-02-05 16:01:00', '{"utm":"referral"}'),
  (23, 7, 'view_item',      '2021-02-05 16:02:00', '{"sku":"B222"}'),
  (24, 7, 'add_to_cart',    '2021-02-05 16:03:00', '{"sku":"B222"}'),
  (25, 7, 'begin_checkout', '2021-02-05 16:04:00', '{"cart_value":58.0}'),
  (26, 7, 'purchase',       '2021-02-05 16:05:00', '{"order_id":"O-0007","value":58.0}'),

  (27, 8, 'session_start',  '2021-02-09 10:31:00', '{"utm":"organic"}'),
  (28, 8, 'view_item',      '2021-02-09 10:32:00', '{"sku":"C333"}'),
  (29, 8, 'add_to_cart',    '2021-02-09 10:34:00', '{"sku":"C333"}'),
  -- Usuario 8 abandona el carrito

  (30, 9, 'session_start',  '2021-02-12 21:46:00', '{"utm":"email"}'),
  -- Usuario 9 solo inicia sesión

  (31, 10, 'session_start',  '2021-02-15 07:51:00', '{"utm":"ads"}'),
  (32, 10, 'view_item',      '2021-02-15 07:52:00', '{"sku":"A101"}'),
  (33, 10, 'add_to_cart',    '2021-02-15 07:53:00', '{"sku":"A101"}'),
  (34, 10, 'begin_checkout', '2021-02-15 07:54:00', '{"cart_value":29.9}'),
  (35, 10, 'purchase',       '2021-02-15 07:56:00', '{"order_id":"O-0010","value":29.9}');

-- Marzo (usuarios 11..15) — actividad más ligera
INSERT INTO events (event_id, user_id, event_name, event_ts, props) VALUES
  (36, 11, 'session_start',  '2021-03-01 10:05:00', '{"utm":"organic"}'),
  (37, 11, 'view_item',      '2021-03-01 10:08:00', '{"sku":"A101"}'),
  (38, 11, 'add_to_cart',    '2021-03-01 10:10:00', '{"sku":"A101"}'),
  (39, 11, 'begin_checkout', '2021-03-01 10:12:00', '{"cart_value":29.9}'),
  (40, 11, 'purchase',       '2021-03-01 10:15:00', '{"order_id":"O-0011","value":29.9}'),

  (41, 12, 'session_start',  '2021-03-03 12:35:00', '{"utm":"ads"}'),
  (42, 12, 'view_item',      '2021-03-03 12:37:00', '{"sku":"B222"}'),

  (43, 13, 'session_start',  '2021-03-08 09:20:00', '{"utm":"referral"}'),
  (44, 13, 'view_item',      '2021-03-08 09:22:00', '{"sku":"C333"}'),
  (45, 13, 'add_to_cart',    '2021-03-08 09:24:00', '{"sku":"C333"}'),

  (46, 14, 'session_start',  '2021-03-12 14:05:00', '{"utm":"email"}'),
  (47, 14, 'view_item',      '2021-03-12 14:07:00', '{"sku":"D444"}'),
  (48, 14, 'add_to_cart',    '2021-03-12 14:09:00', '{"sku":"D444"}'),
  (49, 14, 'begin_checkout', '2021-03-12 14:11:00', '{"cart_value":45.0}'),
  (50, 14, 'purchase',       '2021-03-12 14:13:00', '{"order_id":"O-0014","value":45.0}'),

  (51, 15, 'session_start',  '2021-03-18 17:50:00', '{"utm":"organic"}'),
  (52, 15, 'view_item',      '2021-03-18 17:52:00', '{"sku":"E555"}'),
  (53, 15, 'add_to_cart',    '2021-03-18 17:54:00', '{"sku":"E555"}');

-- Eventos de retención: usuarios que regresan semanas después
INSERT INTO events (event_id, user_id, event_name, event_ts, props) VALUES
  -- Usuario 1 regresa en semana 1 y semana 3
  (54, 1, 'session_start', '2021-01-09 10:00:00', '{"utm":"direct"}'),
  (55, 1, 'view_item',     '2021-01-09 10:05:00', '{"sku":"F666"}'),
  (56, 1, 'session_start', '2021-01-23 15:00:00', '{"utm":"email"}'),

  -- Usuario 2 regresa en semana 2
  (57, 2, 'session_start', '2021-01-17 12:00:00', '{"utm":"ads"}'),
  (58, 2, 'view_item',     '2021-01-17 12:05:00', '{"sku":"A101"}'),
  (59, 2, 'purchase',      '2021-01-17 12:10:00', '{"order_id":"O-0002b","value":29.9}'),

  -- Usuario 5 regresa en semana 1
  (60, 5, 'session_start', '2021-01-19 09:00:00', '{"utm":"organic"}'),

  -- Usuario 7 regresa en semana 1 y semana 2
  (61, 7, 'session_start', '2021-02-12 11:00:00', '{"utm":"referral"}'),
  (62, 7, 'view_item',     '2021-02-12 11:05:00', '{"sku":"A101"}'),
  (63, 7, 'session_start', '2021-02-20 14:00:00', '{"utm":"direct"}'),

  -- Usuario 10 regresa en semana 1
  (64, 10, 'session_start', '2021-02-22 08:00:00', '{"utm":"ads"}'),
  (65, 10, 'view_item',     '2021-02-22 08:05:00', '{"sku":"B222"}'),

  -- Usuario 11 regresa en semana 1
  (66, 11, 'session_start', '2021-03-08 11:00:00', '{"utm":"organic"}'),
  (67, 11, 'view_item',     '2021-03-08 11:05:00', '{"sku":"D444"}');
```

### 0.3 Verificación rápida

Ejecuta estas consultas para confirmar que los datos se cargaron correctamente:

```sql
-- Verificar usuarios
SELECT COUNT(*) AS total_users FROM users;
-- Esperado: 15

-- Verificar eventos
SELECT COUNT(*) AS total_events FROM events;
-- Esperado: 67

-- Vista rápida de la distribución de eventos
SELECT event_name, COUNT(*) AS cantidad
FROM events
GROUP BY event_name
ORDER BY cantidad DESC;
```

---
# Bloque 1 · ¿Qué es el Data Journey? (15 min)

## 1.1 Definición del Data Journey (User Journey)

Un **Data Journey** (o User Journey) es la representación del recorrido completo que realiza un usuario dentro de un producto digital, desde su primer contacto hasta su conversión y retención posterior.

**¿Por qué importa?**

Cuando analizamos datos de un producto digital, no basta con mirar métricas aisladas. Necesitamos entender el **flujo completo** por el que pasa un usuario. Esto nos permite:

- **Identificar dónde se pierden usuarios** (puntos de fricción).
- **Medir la efectividad** de cada etapa del proceso.
- **Priorizar mejoras** basándonos en datos, no en intuición.
- **Conectar métricas de producto con impacto de negocio** (conversión, LTV, retención).

> En pocas palabras: el Data Journey convierte una masa de eventos sin contexto en una **historia con estructura** que podemos analizar y optimizar.

## 1.2 Etapas típicas de un Data Journey

Un journey se divide en **etapas secuenciales** que representan las acciones clave del usuario. Cada industria tiene su propio journey, pero la estructura es similar:

### E-commerce (nuestro caso de estudio)
| Etapa | Evento | ¿Qué significa? |
|-------|--------|-----------------|
| 1. Descubrimiento | `session_start` | El usuario llega al sitio |
| 2. Exploración | `view_item` | Mira productos |
| 3. Intención | `add_to_cart` | Agrega al carrito |
| 4. Compromiso | `begin_checkout` | Inicia el pago |
| 5. Conversión | `purchase` | Completa la compra |

### Educación online
| Etapa | Evento | ¿Qué significa? |
|-------|--------|-----------------|
| 1. Descubrimiento | `landing_visit` | Llega a la landing page |
| 2. Registro | `signup` | Crea una cuenta |
| 3. Activación | `first_lesson` | Toma la primera lección |
| 4. Engagement | `complete_module` | Termina un módulo |
| 5. Conversión | `certification` | Obtiene certificado |

### Servicios públicos digitales
| Etapa | Evento | ¿Qué significa? |
|-------|--------|-----------------|
| 1. Acceso | `portal_login` | Ingresa al portal |
| 2. Solicitud | `request_appointment` | Solicita turno |
| 3. Documentación | `upload_docs` | Adjunta documentos |
| 4. Pago | `payment` | Realiza el pago |
| 5. Confirmación | `confirmation` | Trámite confirmado |

> **Concepto clave:** Cada journey es un **embudo (funnel)**: muchos usuarios entran en la etapa 1, y en cada paso posterior se van perdiendo algunos. Entender *dónde* y *por qué* se pierden es el objetivo central del análisis.

## 1.3 Métricas del Data Journey

Las métricas que calculamos a lo largo del journey son:

**Tasa de conversión (Conversion Rate):** porcentaje de usuarios que pasan de una etapa a la siguiente (o de la primera a la última).

**Drop-off:** cantidad de usuarios que se pierden entre una etapa y la siguiente.

**Tiempo entre etapas:** cuánto tarda un usuario en avanzar (puede indicar fricción).

**Retención:** ¿los usuarios que convirtieron regresan después? (se analiza con cohorts).

> Estas métricas son las que conectan el journey con decisiones de negocio: "¿dónde invertir para mejorar la conversión?" o "¿qué segmentos retienen mejor?".

## 1.4 Campos clave en la base de datos de eventos

Para reconstruir un journey desde datos, necesitamos al menos estos campos:

| Campo | Propósito | Ejemplo |
|-------|-----------|---------|
| `user_id` | Identificar al usuario | `1`, `2`, `3` |
| `event_ts` | Orden temporal de los eventos | `'2021-01-02 09:12:00'` |
| `event_name` | Tipo de evento (etapa del journey) | `'session_start'`, `'purchase'` |
| `props` | Metadatos adicionales (SKU, valor, UTM) | `'{"sku":"A101","value":29.9}'` |

**Exploración inicial de nuestra tabla:**

```sql
-- ¿Qué eventos distintos tenemos?
SELECT DISTINCT event_name
FROM events
ORDER BY event_name;

-- ¿Cuántos usuarios únicos hay?
SELECT COUNT(DISTINCT user_id) AS usuarios_unicos
FROM events;

-- Vista general: primeros 10 eventos ordenados cronológicamente
SELECT *
FROM events
ORDER BY event_ts
LIMIT 10;
```

## 1.5 Validación de calidad de datos

Antes de construir cualquier análisis del journey, **siempre** debemos validar los datos. Tres chequeos fundamentales:

### 1) Buscar duplicados exactos
```sql
SELECT user_id, event_name, event_ts, COUNT(*) AS n
FROM events
GROUP BY user_id, event_name, event_ts
HAVING COUNT(*) > 1
ORDER BY n DESC;
```

### 2) Detectar eventos fuera de rango temporal
```sql
SELECT *
FROM events
WHERE event_ts > CURRENT_TIMESTAMP
   OR event_ts < '2020-01-01';
```

### 3) Usuarios con carrito pero sin checkout ni compra (journey incompleto)
```sql
SELECT DISTINCT e.user_id,
       MAX(CASE WHEN e.event_name = 'add_to_cart'    THEN 1 ELSE 0 END) AS tiene_carrito,
       MAX(CASE WHEN e.event_name = 'begin_checkout' THEN 1 ELSE 0 END) AS tiene_checkout,
       MAX(CASE WHEN e.event_name = 'purchase'       THEN 1 ELSE 0 END) AS tiene_compra
FROM events e
GROUP BY e.user_id
HAVING tiene_carrito = 1 AND tiene_compra = 0
ORDER BY e.user_id;
```

> **Documenta siempre** tus hallazgos de calidad antes de avanzar al análisis. Un funnel construido sobre datos sucios genera conclusiones incorrectas.

---
# Bloque 2 · ¿Qué son las CTEs? (20 min)

## 2.1 Definición y motivación

Una **CTE** (Common Table Expression) es una consulta temporal con nombre que existe solo durante la ejecución de la sentencia SQL que la contiene. Se define con la cláusula `WITH`.

### ¿Por qué usar CTEs?

**Sin CTEs** (subconsultas anidadas):
```sql
SELECT *
FROM (
    SELECT *
    FROM (
        SELECT user_id, event_name
        FROM events
        WHERE event_name = 'purchase'
    ) AS compras
    WHERE user_id IN (
        SELECT user_id
        FROM users
        WHERE plan = 'paid'
    )
) AS resultado;
```

**Con CTEs** (paso a paso, legible):
```sql
WITH compras AS (
    SELECT user_id, event_name
    FROM events
    WHERE event_name = 'purchase'
),
usuarios_paid AS (
    SELECT user_id
    FROM users
    WHERE plan = 'paid'
)
SELECT c.*
FROM compras c
INNER JOIN usuarios_paid u ON c.user_id = u.user_id;
```

### Ventajas de las CTEs:
- **Legibilidad:** Cada paso tiene un nombre descriptivo.
- **Modularidad:** Puedes reutilizar una CTE varias veces en la misma consulta.
- **Debugging:** Puedes ejecutar cada CTE por separado para verificar resultados intermedios.
- **Mantenimiento:** Es más fácil modificar una parte sin romper el resto.

## 2.2 Sintaxis de las CTEs

La estructura general es:

```sql
WITH nombre_cte_1 AS (
    -- Primera consulta
    SELECT ...
    FROM ...
),
nombre_cte_2 AS (
    -- Segunda consulta (puede usar nombre_cte_1)
    SELECT ...
    FROM nombre_cte_1
    WHERE ...
)
-- Consulta final que usa las CTEs
SELECT ...
FROM nombre_cte_2;
```

**Reglas importantes:**
- La palabra `WITH` solo aparece **una vez**, al inicio.
- Cada CTE se separa con **coma** (`,`).
- La **última CTE no lleva coma** — le sigue directamente el `SELECT` final.
- Una CTE puede **referenciar a las CTEs definidas antes** de ella.
- Las CTEs **no existen** fuera de la sentencia donde se definieron.

## 2.3 Ejercicios progresivos de CTEs

---

### Ejercicio CTE-1: Tu primera CTE (Nivel: ⭐ Básico)

**Descripción:** Crea una CTE que seleccione solo los usuarios con plan `'free'` y luego cuenta cuántos hay.

**Objetivo:** Entender la sintaxis básica de `WITH ... AS (...)` y cómo se referencia en el SELECT final.

**Explicación:** Una CTE actúa como una "tabla virtual temporal". Primero defines qué datos contiene, y luego la usas en tu consulta final como si fuera una tabla real.

**Pista:** Define una CTE llamada `usuarios_free` que filtre la tabla `users` por `plan = 'free'`. Luego, en el SELECT final, haz un `COUNT(*)` sobre esa CTE.

---

**✅ Solución:**
```sql
WITH usuarios_free AS (
    SELECT user_id, signup_ts, plan, channel, device
    FROM users
    WHERE plan = 'free'
)
SELECT COUNT(*) AS total_free_users
FROM usuarios_free;
```

> **Resultado esperado:** 9 usuarios free (de los 15 totales).

### Ejercicio CTE-2: Dos CTEs encadenadas (Nivel: ⭐ Básico)

**Descripción:** Crea una CTE con los usuarios `'paid'` y otra CTE con los eventos de tipo `'purchase'`. Luego, cruza ambas para encontrar qué usuarios de pago hicieron al menos una compra.

**Objetivo:** Practicar el uso de múltiples CTEs separadas por coma y combinarlas con un JOIN.

**Explicación:** Cuando tienes dos conjuntos de datos que necesitas cruzar, puedes definir cada uno como una CTE independiente y luego hacer el JOIN en la consulta final. Esto es mucho más claro que anidar subconsultas.

**Pista:** La primera CTE filtra `users` por `plan = 'paid'`. La segunda filtra `events` por `event_name = 'purchase'`. En el SELECT final, haz un `INNER JOIN` entre ambas por `user_id`.

---

**✅ Solución:**
```sql
WITH usuarios_paid AS (
    SELECT user_id, plan, channel
    FROM users
    WHERE plan = 'paid'
),
compras AS (
    SELECT DISTINCT user_id
    FROM events
    WHERE event_name = 'purchase'
)
SELECT up.user_id, up.channel
FROM usuarios_paid up
INNER JOIN compras c ON up.user_id = c.user_id;
```

> **Resultado esperado:** Usuarios 2, 7, 10 y 14 (los 4 usuarios paid que compraron).

### Ejercicio CTE-3: CTE que referencia otra CTE (Nivel: ⭐⭐ Intermedio)

**Descripción:** Crea tres CTEs encadenadas:
1. `todos_los_eventos`: filtra eventos de enero 2021.
2. `eventos_por_usuario`: cuenta cuántos eventos tiene cada usuario en ese mes.
3. `usuarios_activos`: filtra solo los usuarios con 3 o más eventos.

Finalmente, muestra los usuarios activos con su conteo.

**Objetivo:** Demostrar que una CTE puede usar datos de una CTE anterior, creando un pipeline de transformaciones.

**Explicación:** Las CTEs se evalúan en orden. Cada CTE solo puede referenciar las que fueron definidas *antes* de ella. Esto permite construir transformaciones paso a paso, donde cada paso es más refinado que el anterior.

**Pista:** En la CTE `eventos_por_usuario`, haz un `GROUP BY user_id` con `COUNT(*)` sobre `todos_los_eventos`. En `usuarios_activos`, filtra con `HAVING` o con `WHERE num_eventos >= 3`.

---

**✅ Solución:**
```sql
WITH todos_los_eventos AS (
    SELECT *
    FROM events
    WHERE event_ts >= '2021-01-01'
      AND event_ts <  '2021-02-01'
),
eventos_por_usuario AS (
    SELECT user_id, COUNT(*) AS num_eventos
    FROM todos_los_eventos
    GROUP BY user_id
),
usuarios_activos AS (
    SELECT *
    FROM eventos_por_usuario
    WHERE num_eventos >= 3
)
SELECT *
FROM usuarios_activos
ORDER BY num_eventos DESC;
```

> **Resultado esperado:** Usuarios 1, 2 y 3 (cada uno con 4-5 eventos en enero). Los usuarios 4 y 5 tienen menos de 3 eventos.

### Ejercicio CTE-4: Reutilizar una CTE múltiples veces (Nivel: ⭐⭐ Intermedio)

**Descripción:** Crea una CTE llamada `resumen_usuario` que para cada usuario calcule: su número total de eventos y si tiene al menos un evento `'purchase'`. Luego, en la consulta final, muestra el promedio de eventos para compradores vs. no compradores.

**Objetivo:** Demostrar que una misma CTE puede usarse varias veces en la consulta final (o en CTEs posteriores), lo cual es una ventaja clave frente a las subconsultas.

**Explicación:** A diferencia de una subconsulta que se "recalcula" cada vez que la usas, una CTE se define una vez y se referencia por nombre cuantas veces necesites. Esto no solo es más legible, sino que evita repetir lógica.

**Pista:** En `resumen_usuario`, usa `MAX(CASE WHEN event_name = 'purchase' THEN 1 ELSE 0 END)` para crear una columna `es_comprador`. Luego agrupa por `es_comprador` en el SELECT final.

---

**✅ Solución:**
```sql
WITH resumen_usuario AS (
    SELECT
        e.user_id,
        COUNT(*)                                                    AS total_eventos,
        MAX(CASE WHEN e.event_name = 'purchase' THEN 1 ELSE 0 END) AS es_comprador
    FROM events e
    GROUP BY e.user_id
)
SELECT
    CASE WHEN es_comprador = 1 THEN 'Comprador' ELSE 'No comprador' END AS segmento,
    COUNT(*)                     AS num_usuarios,
    ROUND(AVG(total_eventos), 2) AS promedio_eventos
FROM resumen_usuario
GROUP BY es_comprador
ORDER BY es_comprador DESC;
```

> **Resultado esperado:** Los compradores tienen un promedio de eventos más alto que los no compradores, lo cual tiene sentido porque recorrieron más etapas del funnel.

### Ejercicio CTE-5: CTE con JOIN a tabla original (Nivel: ⭐⭐⭐ Avanzado)

**Descripción:** Crea una CTE que identifique el primer evento de cada usuario (su primera interacción con el producto). Luego, haz un JOIN con la tabla `users` para mostrar: `user_id`, `plan`, `channel`, `primer_evento` (nombre) y `fecha_primer_evento`.

**Objetivo:** Combinar CTEs con funciones de ventana (`ROW_NUMBER`) y JOINs, una técnica muy usada en análisis de journey.

**Explicación:** Para encontrar el "primer evento" de cada usuario, usamos `ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_ts)` y filtramos donde `rn = 1`. Esto nos da exactamente una fila por usuario con su evento más antiguo.

**Pista:** La CTE debe incluir `ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY event_ts) AS rn`. Luego filtra `WHERE rn = 1`. En el SELECT final, cruza con `users` por `user_id`.

---

**✅ Solución:**
```sql
WITH primer_evento AS (
    SELECT
        user_id,
        event_name,
        event_ts,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_ts) AS rn
    FROM events
)
SELECT
    u.user_id,
    u.plan,
    u.channel,
    pe.event_name  AS primer_evento,
    pe.event_ts    AS fecha_primer_evento
FROM users u
INNER JOIN primer_evento pe
    ON u.user_id = pe.user_id
   AND pe.rn = 1
ORDER BY u.user_id;
```

> **Resultado esperado:** Todos los usuarios con `session_start` como su primer evento, ordenados del 1 al 15.

---
# Bloque 3 · Construcción de Funnels de conversión (20 min)

## 3.1 ¿Qué es un funnel?

Un **funnel de conversión** es la visualización del journey como un embudo: mostramos cuántos usuarios pasan por cada etapa y cuántos se pierden en el camino.

Nuestro funnel de e-commerce tiene 5 etapas:
```
session_start → view_item → add_to_cart → begin_checkout → purchase
```

El objetivo es responder:
- ¿Cuántos usuarios llegan a cada etapa?
- ¿Dónde está el mayor **drop-off** (caída)?
- ¿Cuál es la **tasa de conversión** total (session → purchase)?

### Ejercicio F-1: Funnel básico con CTEs (Nivel: ⭐ Básico)

**Descripción:** Construye un funnel para TODOS los datos (ene-mar 2021). Usa una CTE por etapa para contar usuarios distintos. Muestra una fila con el conteo de cada etapa.

**Objetivo:** Aplicar CTEs para segmentar usuarios por etapa del journey y obtener el conteo por etapa.

**Explicación:** Cada CTE selecciona los `user_id` distintos que realizaron un evento específico. La consulta final usa subconsultas escalares `(SELECT COUNT(*) FROM cte)` para reunir todos los conteos en una sola fila.

**Pista:** Crea 5 CTEs (`cte_session`, `cte_view`, `cte_cart`, `cte_checkout`, `cte_purchase`), cada una con `SELECT DISTINCT user_id FROM events WHERE event_name = '...'`. Al final, haz un SELECT con un COUNT de cada CTE.

---

**✅ Solución:**
```sql
WITH cte_session AS (
    SELECT DISTINCT user_id FROM events WHERE event_name = 'session_start'
),
cte_view AS (
    SELECT DISTINCT user_id FROM events WHERE event_name = 'view_item'
),
cte_cart AS (
    SELECT DISTINCT user_id FROM events WHERE event_name = 'add_to_cart'
),
cte_checkout AS (
    SELECT DISTINCT user_id FROM events WHERE event_name = 'begin_checkout'
),
cte_purchase AS (
    SELECT DISTINCT user_id FROM events WHERE event_name = 'purchase'
)
SELECT
    (SELECT COUNT(*) FROM cte_session)  AS session_users,
    (SELECT COUNT(*) FROM cte_view)     AS view_users,
    (SELECT COUNT(*) FROM cte_cart)     AS cart_users,
    (SELECT COUNT(*) FROM cte_checkout) AS checkout_users,
    (SELECT COUNT(*) FROM cte_purchase) AS purchase_users;
```

> **Resultado esperado:** 15 → 14 → 10 → 6 → 5 (los números exactos del embudo).

### Ejercicio F-2: Funnel con drop-off y conversión (Nivel: ⭐⭐ Intermedio)

**Descripción:** Extiende el funnel anterior para calcular:
- El **drop-off** (usuarios perdidos) entre cada par de etapas consecutivas.
- La **tasa de conversión global** de `session_start` a `purchase` como porcentaje.

**Objetivo:** Calcular métricas de negocio derivadas del funnel usando aritmética entre columnas.

**Explicación:** El drop-off entre dos etapas es simplemente la resta: `usuarios_etapa_N - usuarios_etapa_N+1`. La tasa de conversión global es `(purchase_users / session_users) * 100`. Usamos `NULLIF` para evitar divisiones por cero.

**Pista:** Coloca todos los conteos en una CTE `counts` y luego calcula las restas y el porcentaje en el SELECT final. Usa `ROUND(100.0 * n_purchase / NULLIF(n_session, 0), 2)`.

---

**✅ Solución:**
```sql
WITH cte_session AS (
    SELECT DISTINCT user_id FROM events WHERE event_name = 'session_start'
),
cte_view AS (
    SELECT DISTINCT user_id FROM events WHERE event_name = 'view_item'
),
cte_cart AS (
    SELECT DISTINCT user_id FROM events WHERE event_name = 'add_to_cart'
),
cte_checkout AS (
    SELECT DISTINCT user_id FROM events WHERE event_name = 'begin_checkout'
),
cte_purchase AS (
    SELECT DISTINCT user_id FROM events WHERE event_name = 'purchase'
),
counts AS (
    SELECT
        (SELECT COUNT(*) FROM cte_session)  AS n_session,
        (SELECT COUNT(*) FROM cte_view)     AS n_view,
        (SELECT COUNT(*) FROM cte_cart)     AS n_cart,
        (SELECT COUNT(*) FROM cte_checkout) AS n_checkout,
        (SELECT COUNT(*) FROM cte_purchase) AS n_purchase
)
SELECT
    n_session,
    n_view,
    n_cart,
    n_checkout,
    n_purchase,
    -- Drop-off entre etapas
    (n_session  - n_view)     AS drop_session_to_view,
    (n_view     - n_cart)     AS drop_view_to_cart,
    (n_cart     - n_checkout) AS drop_cart_to_checkout,
    (n_checkout - n_purchase) AS drop_checkout_to_purchase,
    -- Tasa de conversión global
    ROUND(100.0 * n_purchase / NULLIF(n_session, 0), 2) AS conversion_rate_pct
FROM counts;
```

> **Resultado esperado:** La mayor caída es de `view_item` a `add_to_cart` (4 usuarios perdidos). La conversión global es ~33.33%.

### Ejercicio F-3: Funnel filtrado por mes (Nivel: ⭐⭐ Intermedio)

**Descripción:** Construye el funnel solo para **febrero 2021** (usuarios 6-10). Calcula conteos por etapa, drop-off y conversión.

**Objetivo:** Practicar filtrado temporal dentro de CTEs y comparar funnels de diferentes periodos.

**Explicación:** Agrega una CTE `base` que filtre los eventos por rango de fechas. Todas las CTEs de etapas deben usar `base` en lugar de `events` directamente. Esto asegura que el funnel solo refleje actividad del periodo seleccionado.

**Pista:** Crea una CTE `base` con `WHERE event_ts >= '2021-02-01' AND event_ts < '2021-03-01'`. Las demás CTEs hacen `SELECT DISTINCT user_id FROM base WHERE event_name = '...'`.

---

**✅ Solución:**
```sql
WITH base AS (
    SELECT *
    FROM events
    WHERE event_ts >= '2021-02-01'
      AND event_ts <  '2021-03-01'
),
cte_session AS (
    SELECT DISTINCT user_id FROM base WHERE event_name = 'session_start'
),
cte_view AS (
    SELECT DISTINCT user_id FROM base WHERE event_name = 'view_item'
),
cte_cart AS (
    SELECT DISTINCT user_id FROM base WHERE event_name = 'add_to_cart'
),
cte_checkout AS (
    SELECT DISTINCT user_id FROM base WHERE event_name = 'begin_checkout'
),
cte_purchase AS (
    SELECT DISTINCT user_id FROM base WHERE event_name = 'purchase'
),
counts AS (
    SELECT
        (SELECT COUNT(*) FROM cte_session)  AS n_session,
        (SELECT COUNT(*) FROM cte_view)     AS n_view,
        (SELECT COUNT(*) FROM cte_cart)     AS n_cart,
        (SELECT COUNT(*) FROM cte_checkout) AS n_checkout,
        (SELECT COUNT(*) FROM cte_purchase) AS n_purchase
)
SELECT
    'Febrero 2021' AS periodo,
    n_session, n_view, n_cart, n_checkout, n_purchase,
    (n_session - n_view)     AS drop_session_to_view,
    (n_view - n_cart)        AS drop_view_to_cart,
    (n_cart - n_checkout)    AS drop_cart_to_checkout,
    (n_checkout - n_purchase)AS drop_checkout_to_purchase,
    ROUND(100.0 * n_purchase / NULLIF(n_session, 0), 2) AS conversion_rate_pct
FROM counts;
```

> **Resultado esperado:** En febrero: 5 → 4 → 3 → 2 → 2. Mayor drop-off en session→view y view→cart. Conversión ~40%.

### 3.2 Interpretación de resultados del funnel

Después de calcular el funnel, el análisis de negocio consiste en:

1. **Identificar el cuello de botella:** ¿Dónde está la mayor caída? Si la mayoría de los usuarios abandona entre `view_item` y `add_to_cart`, podría indicar que los productos no son atractivos, los precios son altos, o falta un CTA claro.

2. **Formular hipótesis:** Basándote en los datos, propón explicaciones. Por ejemplo:
   - "Los usuarios móviles abandonan más en checkout → posible problema de UX móvil."
   - "Los usuarios de ads tienen menor conversión → las expectativas de la publicidad no coinciden con el producto."

3. **Proponer acciones:** Cada hipótesis debe llevar a una acción medible:
   - Rediseñar la página de checkout para móvil.
   - Ajustar el copy de los anuncios.
   - Implementar un A/B test para validar.

> **Ejercicio discursivo:** Observa los resultados del funnel que calculaste. ¿Cuál es el mayor cuello de botella? Propón una hipótesis y una acción concreta para reducir el drop-off.

---
# Bloque 4 · Análisis de retención con Cohorts (20 min)

## 4.1 ¿Qué es un cohort?

Un **cohort** es un grupo de usuarios que comparten una característica en común, generalmente su **fecha de registro** (signup). Los agrupamos por mes (o semana) y luego medimos su comportamiento en periodos posteriores.

**Pregunta que responde:**
> "De los usuarios que se registraron en enero, ¿cuántos regresaron en la semana 1? ¿Y en la semana 2?"

Esto nos da una **tabla de retención** que muestra cómo se "desvanece" cada grupo con el tiempo.

### ¿Por qué cohorts?
- Permiten comparar la salud del producto entre diferentes periodos de adquisición.
- Detectan si las mejoras al producto realmente mejoran la retención de nuevos usuarios.
- Conectan con el funnel: ¿los cohorts más recientes convierten mejor?

### Ejercicio C-1: Identificar cohorts mensuales (Nivel: ⭐ Básico)

**Descripción:** Crea una consulta que muestre cada usuario con su **cohort mensual** (el mes de su signup) y cuente cuántos usuarios hay por cohort.

**Objetivo:** Aprender a extraer el mes de una fecha para agrupar usuarios por cohort.

**Explicación:** En DuckDB, `DATE_TRUNC('month', signup_ts)` toma un timestamp y lo convierte al primer día de su mes. Esto nos permite agrupar por mes sin preocuparnos por los días exactos.

**Pista:** Usa `DATE_TRUNC('month', signup_ts) AS cohort_month` en un SELECT sobre `users`, y luego agrupa por `cohort_month`.

---

**✅ Solución:**
```sql
SELECT
    DATE_TRUNC('month', signup_ts) AS cohort_month,
    COUNT(*) AS cohort_size
FROM users
GROUP BY cohort_month
ORDER BY cohort_month;
```

> **Resultado esperado:** Enero: 5 usuarios, Febrero: 5 usuarios, Marzo: 5 usuarios.

### Ejercicio C-2: Tabla de retención semanal (Nivel: ⭐⭐⭐ Avanzado)

**Descripción:** Construye una tabla de retención que muestre, para cada cohort mensual, el porcentaje de usuarios que tuvieron al menos un evento en las semanas 0, 1, 2 y 3 después de su mes de registro.

**Objetivo:** Construir una tabla de retención completa usando CTEs, CROSS JOIN y lógica de ventanas temporales.

**Explicación:**
- **Paso 1:** Asignar cada usuario a su cohort mensual.
- **Paso 2:** Generar un grid de semanas (0, 1, 2, 3) con `UNNEST`.
- **Paso 3:** Para cada (cohort, semana), contar cuántos usuarios tuvieron eventos en esa ventana temporal.
- **Paso 4:** Calcular el porcentaje sobre el total del cohort.

La semana `w` se define como los días `[7*w, 7*(w+1))` desde el inicio del cohort.

**Pista:** Usa `UNNEST(ARRAY[0,1,2,3]) AS week_offset` para generar el grid. Luego haz un CROSS JOIN con los usuarios y filtra los eventos cuya diferencia de días caiga en el rango `[7*w, 7*(w+1))`.

---

**✅ Solución:**
```sql
WITH
-- 1) Asignar cohort mensual a cada usuario
u AS (
    SELECT
        user_id,
        DATE_TRUNC('month', signup_ts) AS cohort_month
    FROM users
),

-- 2) Grid de semanas 0..3
grid AS (
    SELECT UNNEST(ARRAY[0, 1, 2, 3]) AS week_offset
),

-- 3) Retención por (cohort, semana)
retention AS (
    SELECT
        u.cohort_month,
        g.week_offset,
        COUNT(DISTINCT CASE
            WHEN EXISTS (
                SELECT 1
                FROM events e
                WHERE e.user_id = u.user_id
                  AND DATEDIFF('day', u.cohort_month, CAST(e.event_ts AS DATE)) >= 7 * g.week_offset
                  AND DATEDIFF('day', u.cohort_month, CAST(e.event_ts AS DATE)) <  7 * (g.week_offset + 1)
            )
            THEN u.user_id
        END) AS retained_users,
        COUNT(DISTINCT u.user_id) AS cohort_users
    FROM u
    CROSS JOIN grid g
    GROUP BY u.cohort_month, g.week_offset
)

-- 4) Porcentaje de retención
SELECT
    cohort_month,
    week_offset,
    retained_users,
    cohort_users,
    ROUND(100.0 * retained_users / NULLIF(cohort_users, 0), 2) AS retention_pct
FROM retention
ORDER BY cohort_month, week_offset;
```

> **Resultado esperado:** La semana 0 tendrá retención alta (usuarios activos inmediatamente), y disminuirá en semanas posteriores. Algunos cohorts pueden tener 0% en semanas lejanas si no hay datos de retorno.

### Ejercicio C-3: Tabla pivote de retención para heatmap (Nivel: ⭐⭐⭐ Avanzado)

**Descripción:** Transforma la tabla de retención anterior en formato "ancho" (pivote): una fila por cohort, una columna por semana (w0, w1, w2, w3) con el porcentaje de retención. Este formato es el que usarías para crear un heatmap en Google Sheets o Excel.

**Objetivo:** Practicar la técnica de pivote manual con `MAX(CASE WHEN ...)`, muy común en SQL analítico.

**Explicación:** SQL no tiene una función PIVOT nativa en todos los motores, pero podemos simularla con `MAX(CASE WHEN week_offset = N THEN valor END)` dentro de un `GROUP BY cohort_month`. Cada columna resultante contiene el valor para esa semana específica.

**Pista:** Toma la CTE `retention` del ejercicio anterior y agrega una CTE `base` con el porcentaje. Luego, en el SELECT final, usa `MAX(CASE WHEN week_offset = 0 THEN retention_pct END) AS w0`, etc.

---

**✅ Solución:**
```sql
WITH
u AS (
    SELECT user_id, DATE_TRUNC('month', signup_ts) AS cohort_month
    FROM users
),
grid AS (
    SELECT UNNEST(ARRAY[0, 1, 2, 3]) AS week_offset
),
retention AS (
    SELECT
        u.cohort_month,
        g.week_offset,
        COUNT(DISTINCT CASE
            WHEN EXISTS (
                SELECT 1 FROM events e
                WHERE e.user_id = u.user_id
                  AND DATEDIFF('day', u.cohort_month, CAST(e.event_ts AS DATE)) >= 7 * g.week_offset
                  AND DATEDIFF('day', u.cohort_month, CAST(e.event_ts AS DATE)) <  7 * (g.week_offset + 1)
            )
            THEN u.user_id
        END) AS retained_users,
        COUNT(DISTINCT u.user_id) AS cohort_users
    FROM u CROSS JOIN grid g
    GROUP BY u.cohort_month, g.week_offset
),
base AS (
    SELECT
        cohort_month,
        week_offset,
        ROUND(100.0 * retained_users / NULLIF(cohort_users, 0), 2) AS retention_pct
    FROM retention
)
SELECT
    cohort_month,
    MAX(CASE WHEN week_offset = 0 THEN retention_pct END) AS w0,
    MAX(CASE WHEN week_offset = 1 THEN retention_pct END) AS w1,
    MAX(CASE WHEN week_offset = 2 THEN retention_pct END) AS w2,
    MAX(CASE WHEN week_offset = 3 THEN retention_pct END) AS w3
FROM base
GROUP BY cohort_month
ORDER BY cohort_month;
```

> **Resultado esperado:** Una tabla de 3 filas (ene, feb, mar) × 4 columnas (w0-w3) con porcentajes. Ideal para copiar a Google Sheets y aplicar formato condicional como heatmap.

### Ejercicio C-4: Retención segmentada por plan (Nivel: ⭐⭐⭐ Avanzado)

**Descripción:** Extiende la tabla de retención para segmentarla por `plan` (free vs paid). Muestra el porcentaje de retención para cada combinación de (cohort_month, plan, week_offset).

**Objetivo:** Segmentar la retención por atributos de negocio para detectar diferencias entre segmentos.

**Explicación:** La única diferencia con el ejercicio anterior es que ahora la CTE `u` también incluye la columna `plan`, y todos los `GROUP BY` agregan `plan`. Esto nos permite comparar si los usuarios paid retienen mejor que los free.

**Pista:** Agrega `u.plan` a la CTE `u`, al `GROUP BY` de la CTE `retention`, y al SELECT final. El resto de la lógica es idéntica.

---

**✅ Solución:**
```sql
WITH
u AS (
    SELECT user_id, DATE_TRUNC('month', signup_ts) AS cohort_month, plan
    FROM users
),
grid AS (
    SELECT UNNEST(ARRAY[0, 1, 2, 3]) AS week_offset
),
retention AS (
    SELECT
        u.cohort_month,
        u.plan,
        g.week_offset,
        COUNT(DISTINCT CASE
            WHEN EXISTS (
                SELECT 1 FROM events e
                WHERE e.user_id = u.user_id
                  AND DATEDIFF('day', u.cohort_month, CAST(e.event_ts AS DATE)) >= 7 * g.week_offset
                  AND DATEDIFF('day', u.cohort_month, CAST(e.event_ts AS DATE)) <  7 * (g.week_offset + 1)
            )
            THEN u.user_id
        END) AS retained_users,
        COUNT(DISTINCT u.user_id) AS cohort_users
    FROM u CROSS JOIN grid g
    GROUP BY u.cohort_month, u.plan, g.week_offset
)
SELECT
    cohort_month,
    plan,
    week_offset,
    retained_users,
    cohort_users,
    ROUND(100.0 * retained_users / NULLIF(cohort_users, 0), 2) AS retention_pct
FROM retention
ORDER BY cohort_month, plan, week_offset;
```

> **Resultado esperado:** Tabla con filas para cada (cohort × plan × semana). Compara: ¿los usuarios `paid` tienen mejor retención que los `free`? Formula una hipótesis de negocio basada en los resultados.

---
# Bloque 5 · Ejercicios integradores (15 min)

Estos ejercicios combinan todos los conceptos vistos: Data Journey, CTEs, Funnels y Cohorts.

### Ejercicio I-1: Journey individual de un usuario (Nivel: ⭐ Básico)

**Descripción:** Muestra el journey completo del usuario 1: todos sus eventos ordenados cronológicamente, incluyendo el tiempo transcurrido (en minutos) entre cada evento y el anterior.

**Objetivo:** Reconstruir el journey de un usuario individual y medir tiempos entre etapas usando funciones de ventana.

**Explicación:** `LAG(event_ts) OVER (ORDER BY event_ts)` nos da el timestamp del evento anterior. La diferencia entre el evento actual y el anterior nos da el tiempo entre etapas. En DuckDB, `DATEDIFF('minute', ts_anterior, ts_actual)` calcula esta diferencia.

**Pista:** Usa `LAG(event_ts) OVER (PARTITION BY user_id ORDER BY event_ts)` para obtener el timestamp anterior. Luego calcula la diferencia con `DATEDIFF`.

---

**✅ Solución:**
```sql
WITH journey AS (
    SELECT
        user_id,
        event_name,
        event_ts,
        LAG(event_ts) OVER (PARTITION BY user_id ORDER BY event_ts) AS prev_event_ts
    FROM events
    WHERE user_id = 1
)
SELECT
    user_id,
    event_name,
    event_ts,
    prev_event_ts,
    DATEDIFF('minute', prev_event_ts, event_ts) AS minutos_desde_anterior
FROM journey
ORDER BY event_ts;
```

> **Resultado esperado:** El journey del usuario 1 con 7-8 eventos (incluyendo sus visitas de retorno), mostrando que su primer journey tomó unos 6 minutos de session_start a purchase, y luego regresó días después.

### Ejercicio I-2: Funnel por canal de adquisición (Nivel: ⭐⭐ Intermedio)

**Descripción:** Construye un funnel de conversión (`session_start` → `purchase`) agrupado por el **canal de adquisición** (`channel` en la tabla `users`). Muestra: canal, usuarios en session, usuarios en purchase, y tasa de conversión.

**Objetivo:** Combinar datos de dos tablas (users + events) usando CTEs y JOINs para segmentar el funnel.

**Explicación:** Necesitas cruzar `events` con `users` para obtener el canal de cada usuario. Luego, agrupas por canal y calculas la conversión. Esto revela qué canales atraen usuarios que realmente compran.

**Pista:** Haz un JOIN entre `events` y `users` dentro de la CTE `base`. Luego cuenta usuarios distintos para `session_start` y `purchase` por separado, agrupados por `channel`.

---

**✅ Solución:**
```sql
WITH base AS (
    SELECT e.user_id, e.event_name, u.channel
    FROM events e
    INNER JOIN users u ON e.user_id = u.user_id
),
por_canal AS (
    SELECT
        channel,
        COUNT(DISTINCT CASE WHEN event_name = 'session_start' THEN user_id END) AS session_users,
        COUNT(DISTINCT CASE WHEN event_name = 'purchase'      THEN user_id END) AS purchase_users
    FROM base
    GROUP BY channel
)
SELECT
    channel,
    session_users,
    purchase_users,
    ROUND(100.0 * purchase_users / NULLIF(session_users, 0), 2) AS conversion_rate_pct
FROM por_canal
ORDER BY conversion_rate_pct DESC;
```

> **Resultado esperado:** Cada canal (organic, ads, referral, email) con su conteo y tasa de conversión. Observa qué canal convierte mejor y cuál peor.

### Ejercicio I-3: Usuarios que abandonaron vs. completaron (Nivel: ⭐⭐⭐ Avanzado)

**Descripción:** Clasifica cada usuario en una de estas categorías según hasta dónde avanzó en el journey:
- `'Solo sesión'`: solo tiene `session_start`
- `'Exploró'`: llegó hasta `view_item` pero no más
- `'Carrito'`: llegó hasta `add_to_cart` pero no checkout
- `'Checkout'`: llegó hasta `begin_checkout` pero no compró
- `'Comprador'`: completó `purchase`

Muestra cuántos usuarios hay en cada categoría.

**Objetivo:** Construir una clasificación de usuarios basada en su avance máximo en el journey, usando CTEs y lógica condicional avanzada.

**Explicación:** Primero, para cada usuario, determinamos qué etapas completó usando `MAX(CASE WHEN ...)`. Luego, con una serie de `CASE WHEN` anidados, asignamos la categoría según la etapa máxima alcanzada.

**Pista:** Crea una CTE que para cada `user_id` calcule columnas booleanas: `tiene_session`, `tiene_view`, `tiene_cart`, `tiene_checkout`, `tiene_purchase`. Luego clasifica con `CASE WHEN tiene_purchase = 1 THEN 'Comprador' WHEN tiene_checkout = 1 THEN 'Checkout' ...`.

---

**✅ Solución:**
```sql
WITH etapas_usuario AS (
    SELECT
        user_id,
        MAX(CASE WHEN event_name = 'session_start'  THEN 1 ELSE 0 END) AS tiene_session,
        MAX(CASE WHEN event_name = 'view_item'       THEN 1 ELSE 0 END) AS tiene_view,
        MAX(CASE WHEN event_name = 'add_to_cart'     THEN 1 ELSE 0 END) AS tiene_cart,
        MAX(CASE WHEN event_name = 'begin_checkout'  THEN 1 ELSE 0 END) AS tiene_checkout,
        MAX(CASE WHEN event_name = 'purchase'        THEN 1 ELSE 0 END) AS tiene_purchase
    FROM events
    GROUP BY user_id
),
clasificacion AS (
    SELECT
        user_id,
        CASE
            WHEN tiene_purchase = 1 THEN 'Comprador'
            WHEN tiene_checkout = 1 THEN 'Checkout abandonado'
            WHEN tiene_cart     = 1 THEN 'Carrito abandonado'
            WHEN tiene_view     = 1 THEN 'Solo exploró'
            WHEN tiene_session  = 1 THEN 'Solo sesión'
            ELSE 'Sin actividad'
        END AS categoria
    FROM etapas_usuario
)
SELECT
    categoria,
    COUNT(*) AS num_usuarios,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM clasificacion), 2) AS porcentaje
FROM clasificacion
GROUP BY categoria
ORDER BY
    CASE categoria
        WHEN 'Solo sesión'          THEN 1
        WHEN 'Solo exploró'         THEN 2
        WHEN 'Carrito abandonado'   THEN 3
        WHEN 'Checkout abandonado'  THEN 4
        WHEN 'Comprador'            THEN 5
    END;
```

> **Resultado esperado:** Una tabla con 5 categorías y el conteo/porcentaje de cada una. Los "Compradores" serán el grupo más pequeño (el fondo del embudo).

---
# Cierre (5 min)

## Takeaways de la sesión

1. **El Data Journey** es la representación del recorrido del usuario. Entenderlo es el primer paso antes de cualquier análisis.

2. **Las CTEs** son la herramienta fundamental para organizar consultas complejas: legibles, modulares, reutilizables. Siempre prefiere CTEs sobre subconsultas anidadas.

3. **Los Funnels** cuantifican el journey: miden conversión y drop-off entre etapas. El cuello de botella más grande es donde hay que actuar.

4. **Los Cohorts** miden retención: ¿los usuarios regresan? Segmentar por atributos (plan, canal, device) revela qué segmentos retienen mejor.

5. **La calidad de datos** viene antes del análisis. Validar duplicados, timestamps y journeys incompletos es obligatorio.

### ¿Necesitas ayuda?
Recuerda los canales de ayuda que tenemos disponibles:
- `DATA-CO-LEARNING`: Revisa los horarios de atencion en la guia dele studiante. Recuerda que tenemos horarios de apoyo todos los días para tus dudas puntuales!
- `DA_CONSULTA`: Uuedes publicar tus preguntas sobre el contenido de la plataforma o el proyecto 24/7. Recuerda que en tu pregunta debes de agregar el tag @Dataconsulta.
- `SPRINT FOCUS`: Para los Sprints 1 al 5 tenemos sesiones extra donde abordamos temáticas de cada sprint a rpofundidad. Revisa la guia del estudiante para conocer los horarios!
- `SESIONES 1:1`: Recuerda que puedes agendar sesiones 1:1 con un tutor. Agendalas con antelación y resuelve todas tus dudas, desde temás del proyecto, curso, carrera hasta problemas técnicos.
- `CANAL DE DISCORD DE COMMUNITY`: Recuerda que tu cohorte tiene un canal especial donde puedes compartir cualquier item interesante que quieras mostrar a tus compañeros de curso.

## Reflexión final

- ¿Por qué el análisis de funnels es crítico para productos digitales?
- ¿Qué ventaja tienen las CTEs para organizar tu código SQL?
- ¿Cómo conectarías los resultados del funnel con una decisión de negocio concreta?

## Siguientes pasos

- **Próxima sesión:** Sprint 4 — Práctica de construcción de Funnels en SQL.
- **Tarea:** Visualiza el funnel de un producto que uses (Netflix, Amazon, Spotify). Identifica al menos 4 etapas.
- **Repaso:** Practica escribir CTEs encadenadas. Si puedes escribir 3 CTEs que se referencien entre sí, ya dominas la herramienta.
- **Exportar retención:** Copia la tabla pivote de retención a Google Sheets y aplica formato condicional (escala de color) para crear un heatmap.

## Resumen de ejercicios

| # | Ejercicio | Nivel | Tema |
|---|-----------|-------|------|
| CTE-1 | Primera CTE básica | ⭐ | CTEs |
| CTE-2 | Dos CTEs con JOIN | ⭐ | CTEs |
| CTE-3 | CTEs encadenadas | ⭐⭐ | CTEs |
| CTE-4 | Reutilizar CTE | ⭐⭐ | CTEs |
| CTE-5 | CTE + Window Function | ⭐⭐⭐ | CTEs |
| F-1 | Funnel básico | ⭐ | Funnels |
| F-2 | Funnel con drop-off | ⭐⭐ | Funnels |
| F-3 | Funnel por mes | ⭐⭐ | Funnels |
| C-1 | Cohorts mensuales | ⭐ | Cohorts |
| C-2 | Retención semanal | ⭐⭐⭐ | Cohorts |
| C-3 | Pivote para heatmap | ⭐⭐⭐ | Cohorts |
| C-4 | Retención segmentada | ⭐⭐⭐ | Cohorts |
| I-1 | Journey individual | ⭐ | Integrador |
| I-2 | Funnel por canal | ⭐⭐ | Integrador |
| I-3 | Clasificación de usuarios | ⭐⭐⭐ | Integrador |